Write a Python function that takes two dictionaries representing student grades from two different semesters and produces a merged report showing: combined GPA, grade trend (improving/declining/stable), and subjects common to both semesters. Use defaultdict and dict comprehension.

Idea

Assume the two dictionaries represent subject → grade points (e.g., {"Math": 8, "Physics": 7}).

The function should compute:

Combined GPA
Average of all grades from both semesters.

Grade Trend
Compare the GPA of semester 1 vs semester 2

sem2 > sem1 → improving

sem2 < sem1 → declining

equal → stable

Common Subjects
Subjects appearing in both dictionaries.

We will use:

defaultdict(list) → collect grades from both semesters

dict comprehension → compute average grade per subject

In [3]:
from collections import defaultdict

def student_report(sem1, sem2):
    grades = defaultdict(list)

    # collect grades from both semesters
    for sub, g in sem1.items():
        grades[sub].append(g)

    for sub, g in sem2.items():
        grades[sub].append(g)

    # average grade per subject using dict comprehension
    subject_avg = {sub: sum(vals)/len(vals) for sub, vals in grades.items()}

    # GPA calculations
    gpa1 = sum(sem1.values()) / len(sem1)
    gpa2 = sum(sem2.values()) / len(sem2)

    combined_gpa = (sum(sem1.values()) + sum(sem2.values())) / (len(sem1) + len(sem2))

    # trend
    if gpa2 > gpa1:
        trend = "improving"
    elif gpa2 < gpa1:
        trend = "declining"
    else:
        trend = "stable"

    # common subjects
    common_subjects = set(sem1) & set(sem2)

    return {
        "combined_gpa": combined_gpa,
        "trend": trend,
        "common_subjects": common_subjects,
        "subject_average": subject_avg
    }
    
sem1 = {"Math": 8, "Physics": 7, "CS": 9}
sem2 = {"Math": 9, "Physics": 8, "AI": 10}

print(student_report(sem1, sem2))

{'combined_gpa': 8.5, 'trend': 'improving', 'common_subjects': {'Math', 'Physics'}, 'subject_average': {'Math': 8.5, 'Physics': 7.5, 'CS': 9.0, 'AI': 10.0}}


Example
sem1 = {"Math": 8, "Physics": 7, "CS": 9}
sem2 = {"Math": 9, "Physics": 8, "AI": 10}

print(student_report(sem1, sem2))

Output structure

{
 'combined_gpa': 8.5,
 'trend': 'improving',
 'common_subjects': {'Math', 'Physics'},
 'subject_average': {'Math': 8.5, 'Physics': 7.5, 'CS': 9, 'AI': 10}
}
Why defaultdict is useful here

Without it, we would need:

if sub not in grades:
    grades[sub] = []

defaultdict(list) automatically creates an empty list, making aggregation cleaner.

Where this pattern is used in industry

Similar logic appears in:

Academic analytics systems

Learning management systems (LMS)

Product analytics dashboards

Financial reporting pipelines

The pattern is essentially multi-source data aggregation + trend analysis.

Critical Evaluation

| Issue                         | Severity    | Explanation                                                                                                                                                                          |
| ----------------------------- | ----------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **Missing subjects handling** | 🔴 Critical | If a subject exists in only one semester, it's still included in `subject_avg` but the calculation is correct (single value). However, no explicit handling for mismatched subjects. |
| **No `.get()` usage**         | 🟡 Medium   | Direct dictionary access with `.values()` and `len()` will raise `ZeroDivisionError` if empty                                                                                        |
| **Trend calculation**         | 🟢 Correct  | Logic is sound, but edge case: what if `sem1` is empty? `gpa1` would crash first                                                                                                     |
| **Edge cases**                | 🔴 Critical | Empty dictionaries cause `ZeroDivisionError` on GPA calculations; single semester works but trend is misleading                                                                      |
| **Pythonic style**            | 🟡 Okay     | Could use `statistics.mean()` and better comprehensions; `defaultdict` is good but manual loops are verbose                                                                          |


Key Problems Identified
ZeroDivisionError: Empty sem1 or sem2 crashes the function
Misleading trend: If one semester is empty, trend comparison is invalid
No validation: No type checking or input validation
Inefficient: Calculates sem1.values() sum twice
Missing docstring: No documentation of expected inputs/outputs
Improved Version

In [2]:
from collections import defaultdict
from typing import Dict, Set, Literal, Union, TypedDict
import statistics


class StudentReportResult(TypedDict):
    """Type definition for the return value of student_report."""
    combined_gpa: Union[float, None]
    trend: Literal["improving", "declining", "stable", "insufficient_data"]
    common_subjects: Set[str]
    subject_average: Dict[str, float]
    semester_gpas: Dict[str, Union[float, None]]


def student_report(
    sem1: Dict[str, float],
    sem2: Dict[str, float]
) -> StudentReportResult:
    """
    Calculate comprehensive student performance report across two semesters.
    
    Args:
        sem1: Dictionary mapping subject names to grades for semester 1
        sem2: Dictionary mapping subject names to grades for semester 2
    
    Returns:
        Dictionary containing:
        - combined_gpa: Weighted average GPA across both semesters, or None if no data
        - trend: Performance trend ("improving", "declining", "stable", or "insufficient_data")
        - common_subjects: Set of subjects present in both semesters
        - subject_average: Average grade per subject across available semesters
        - semester_gpas: Individual GPAs for each semester (None if empty)
    
    Raises:
        TypeError: If inputs are not dictionaries or contain non-numeric grades
    
    Examples:
        >>> sem1 = {"Math": 85.0, "Physics": 90.0}
        >>> sem2 = {"Math": 88.0, "Chemistry": 92.0}
        >>> report = student_report(sem1, sem2)
        >>> report["combined_gpa"]
        88.75
    """
    # Input validation
    if not isinstance(sem1, dict) or not isinstance(sem2, dict):
        raise TypeError("Both semesters must be dictionaries")
    
    # Validate all grades are numeric
    all_grades = list(sem1.values()) + list(sem2.values())
    if all_grades and not all(isinstance(g, (int, float)) for g in all_grades):
        raise TypeError("All grades must be numeric (int or float)")
    
    # Collect grades per subject using defaultdict with list comprehension
    grades: defaultdict[str, list[float]] = defaultdict(list)
    
    for sub, g in {**sem1, **sem2}.items():
        # Determine which semester this grade came from
        if sub in sem1:
            grades[sub].append(sem1[sub])
        if sub in sem2 and sub not in sem1 or (sub in sem2 and sem1.get(sub) != sem2.get(sub)):
            # Handle case where subject in both with different values
            if sub in sem2:
                grades[sub].append(sem2[sub])
    
    # Rebuild properly: iterate through each semester separately to preserve duplicates
    grades = defaultdict(list)
    for sub, g in sem1.items():
        grades[sub].append(float(g))
    for sub, g in sem2.items():
        grades[sub].append(float(g))
    
    # Calculate subject averages using statistics.mean for robustness
    subject_avg = {
        sub: statistics.mean(vals) 
        for sub, vals in grades.items()
    } if grades else {}
    
    # Safe GPA calculations with empty dict handling
    def calc_gpa(semester: Dict[str, float]) -> Union[float, None]:
        """Calculate GPA for a semester, returning None if empty."""
        if not semester:
            return None
        return statistics.mean(semester.values())
    
    gpa1 = calc_gpa(sem1)
    gpa2 = calc_gpa(sem2)
    
    # Combined GPA calculation
    total_grades = list(sem1.values()) + list(sem2.values())
    combined_gpa = statistics.mean(total_grades) if total_grades else None
    
    # Trend determination with proper edge case handling
    trend: Literal["improving", "declining", "stable", "insufficient_data"]
    if gpa1 is None or gpa2 is None:
        trend = "insufficient_data"
    elif gpa2 > gpa1:
        trend = "improving"
    elif gpa2 < gpa1:
        trend = "declining"
    else:
        trend = "stable"
    
    # Common subjects using set intersection
    common_subjects = set(sem1.keys()) & set(sem2.keys())
    
    return {
        "combined_gpa": combined_gpa,
        "trend": trend,
        "common_subjects": common_subjects,
        "subject_average": subject_avg,
        "semester_gpas": {"sem1": gpa1, "sem2": gpa2}
    }


# Additional helper function for batch processing
def batch_student_reports(
    students: Dict[str, tuple[Dict[str, float], Dict[str, float]]]
) -> Dict[str, StudentReportResult]:
    """
    Generate reports for multiple students.
    
    Args:
        students: Dictionary mapping student names to (sem1, sem2) tuples
    
    Returns:
        Dictionary mapping student names to their reports
    """
    return {
        name: student_report(sem1, sem2) 
        for name, (sem1, sem2) in students.items()
    }